In [ ]:
%load_ext cudf.pandas

In [ ]:
#
# Import Visualization FrameWork Tools
import pandas as pd
import numpy as np

# import missingno as msno

In [ ]:
%%time
### cell 0 ###

df = pd.read_csv("/home/dias-benchmarks/new_notebooks/nyc-airbnb/AB_NYC_2019.csv")
factor = 30
df = pd.concat([df] * factor)
df.head()

In [ ]:
%%time
### cell 1 ###

df.shape

In [ ]:
%%time
### cell 2 ###

# check null-value and each columns' dtype to do EDA.
print(df.info())

In [ ]:
%%time
### cell 3 ###

df.drop(["id", "last_review", "name", "host_name"], axis=1, inplace=True)
df.head()

In [ ]:
%%time
### cell 4 ###

df = df[df["availability_365"] != 0].reset_index()
df.head()

In [ ]:
%%time
### cell 5 ###

df["reviews_per_month"] = df["reviews_per_month"].fillna(0)
df.head()

In [ ]:
%%time
### cell 6 ###

df.isnull().sum()

In [ ]:
%%time
### cell 7 ###

df.shape

In [ ]:
%%time
### cell 8 ###

nyc_sub_1 = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]

x_1 = nyc_sub_1
y_1 = []
for i_1 in x_1:
    y_1.append(df[df["neighbourhood_group"] == i_1]["price"].values.tolist())
y_1

In [ ]:
%%time
### cell 9 ###

nyc_sub_2 = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]

# Use cudf’s GPU‐accelerated list aggregation
grouped_prices = df.groupby("neighbourhood_group")["price"].agg("collect")

# Reorder on GPU then convert to a Python list for the final output
y_optimized = grouped_prices.loc[nyc_sub_2].tolist()
y_optimized

In [ ]:
%%time
### cell 10 ###

df["price"].values[0]

In [ ]:
%%time
### cell 11 ###

# Optimized cell 12
# Define the bin edges (right=False to match the original < and ≤ logic)
bins = [0, 50, 250, 500, 750, 1000, float("inf")]

# Initialize all groups to 0 (as in the original loop)
df["price_group"] = 0

# Vectorize the bucketing for the first 100 entries only (to match the original slicing)
df["price_group"] = pd.cut(df["price"], bins=bins, labels=False, right=False).astype(
    df["price_group"].dtype
)

In [ ]:
%%time
### cell 12 ###

suburb_area = [59100000, 183400000, 464900000, 109000000, 151500000]
distance_room_to_room = []
for i_2, city_1 in enumerate(nyc_sub_2):
    distance_room_to_room.append(
        (suburb_area[i_2] / len(df[df["neighbourhood_group"] == city_1])) * (1 / 1000)
    )

distance_room_to_room

In [ ]:
%%time
### cell 13 ###

total_amount_sub = []
total_amount_sub_50_250 = []
aver_price_total = []
aver_price_50_250 = []
for i_3, city_2 in enumerate(nyc_sub_2):
    req = df[
        (df["neighbourhood_group"] == city_2) & (df["price"] > 50) & (df["price"] < 250)
    ]
    total_amount_sub.append(len(df[df["neighbourhood_group"] == city_2]))
    total_amount_sub_50_250.append(len(req))
    aver_price_total.append(
        round(np.mean(df[df["neighbourhood_group"] == city_2]["price"]), 3)
    )
    aver_price_50_250.append(round(np.mean(req["price"]), 3))
total_amount_sub, total_amount_sub_50_250, aver_price_total, aver_price_50_250

In [ ]:
%%time
### cell 14 ###

# Compute neighborhood counts for ordering (descending frequency)
counts = df["neighbourhood"].value_counts()
order = counts.index

# Vectorized aggregation of mean price and first neighbourhood_group, then reorder
mean_price = df.groupby("neighbourhood").price.mean()[order]
neigh_group = df.groupby("neighbourhood").neighbourhood_group.first()[order]

# Map neighbourhood groups to colors
ecolor_map = {
    "Manhattan": "red",
    "Brooklyn": "blue",
    "Queens": "green",
    "Bronx": "orange",
    "Staten Island": "purple",
}

# Extract results as lists
nei_x_1 = order.to_list()
nei_y_1 = mean_price.to_list()
colors_1 = neigh_group.map(ecolor_map).to_list()

nei_x_1, nei_y_1, colors_1

In [ ]:
%%time
### cell 15 ###

# compute threshold once (as in the original loop)
threshold = round(np.mean(nei_y_1), 3)

# compute mean price per neighbourhood on the GPU
# (cudf handles data placement automatically via the loaded extension)
group_means = df.groupby("neighbourhood")["price"].mean()

# preserve the same order as the original loop (value_counts order)
neis_order = df["neighbourhood"].value_counts().index
ordered_means = group_means.reindex(neis_order)

# split into high/low relative to the threshold
high_means = ordered_means[ordered_means >= threshold]
low_means = ordered_means[ordered_means < threshold]

# collect the results in lists exactly as the original code did
nei_x_aver_h_1 = high_means.index.tolist()
nei_y_aver_h_1 = high_means.tolist()
nei_x_aver_l_1 = low_means.index.tolist()
nei_y_aver_l_1 = low_means.tolist()

nei_x_aver_h_1, nei_x_aver_l_1, nei_y_aver_h_1, nei_y_aver_l_1

In [ ]:
%%time
### cell 16 ###

colors_2 = []
for neigh_2 in nei_x_aver_h_1:
    if (
        df[df["neighbourhood"] == neigh_2]["neighbourhood_group"].values[0]
        == "Manhattan"
    ):
        colors_2.append("red")
    elif (
        df[df["neighbourhood"] == neigh_2]["neighbourhood_group"].values[0]
        == "Brooklyn"
    ):
        colors_2.append("blue")
    elif (
        df[df["neighbourhood"] == neigh_2]["neighbourhood_group"].values[0] == "Queens"
    ):
        colors_2.append("green")
    elif df[df["neighbourhood"] == neigh_2]["neighbourhood_group"].values[0] == "Bronx":
        colors_2.append("orange")
    elif (
        df[df["neighbourhood"] == neigh_2]["neighbourhood_group"].values[0]
        == "Staten Island"
    ):
        colors_2.append("purple")
colors_2

In [ ]:
%%time
### cell 17 ###

# Vectorized mapping of neighbourhood to colors
group_color = {
    "Manhattan": "red",
    "Brooklyn": "blue",
    "Queens": "green",
    "Bronx": "orange",
    "Staten Island": "purple",
}

# Create a Series of the target neighbourhoods
nei_series = pd.Series(nei_x_aver_h_1)

# Build a GPU mapping from neighbourhood to neighbourhood_group
mapping = (
    df[["neighbourhood", "neighbourhood_group"]]
    .drop_duplicates(subset=["neighbourhood"])
    .set_index("neighbourhood")["neighbourhood_group"]
)

# Map neighbourhoods to groups and then to colors
colors_3 = nei_series.map(mapping).map(group_color).tolist()
colors_3

In [ ]:
%%time
### cell 18 ###

x_2 = df["calculated_host_listings_count"].value_counts().sort_index().index.astype(str)
y_2 = df["calculated_host_listings_count"].value_counts().sort_index().values

x_2, y_2